# 📓 01. Data Collection & Initial Audit
> **Đề tài:** Phân tích Không gian - Kinh tế thị trường phòng trọ TP.HCM & Bình Dương: Mô hình định giá hợp lý và Hệ thống phát hiện tin đăng bất thường trên nền tảng Chợ Tốt  
> **Môn học:** IE313 - Phân tích và Trực quan hóa Dữ liệu (UIT - VNUHCM)  

---
## 🎯 Mục tiêu Notebook:
1. Minh chứng kiến trúc thu thập dữ liệu phân tầng (**3-Stage Stratified API Crawling**) giải quyết giới hạn offset của Chợ Tốt.
2. Kiểm toán chất lượng dữ liệu thô (**Raw Data Audit**): Tỷ lệ khuyết, phân phối diện tích/giá ban đầu, tính toàn vẹn.
3. Lưu trữ minh chứng gốc dạng **JSON Lines (.jsonl)** và tạo bản chụp khử trùng lặp sơ bộ **Parquet (.parquet)**.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập style trực quan hóa
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

BASE_DIR = Path('..').resolve()
RAW_JSONL = BASE_DIR / 'data' / 'raw' / 'chotot_raw_hcm_bd.jsonl'
METADATA_FILE = BASE_DIR / 'data' / 'raw' / 'crawl_metadata.json'
INTERIM_PARQUET = BASE_DIR / 'data' / 'interim' / 'deduplicated_data.parquet'

print('BASE_DIR:', BASE_DIR)

## 1. Kiểm tra Metadata & Quy trình Cào Phân tầng
Chiến lược cào: Lặp qua từng `area_v2` (Quận/Huyện) thuộc TP.HCM (`region_v2=13000`) và Bình Dương (`region_v2=2011`).

In [ ]:
if METADATA_FILE.exists():
    with open(METADATA_FILE, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print('=== CRAWL RUNTIME METADATA ===')
    for k, v in meta.items():
        if k != 'records_per_district':
            print(f' - {k}: {v}')
    
    # Biểu đồ số lượng tin đăng theo từng Quận
    dist_records = meta.get('records_per_district', {})
    if dist_records:
        df_dist = pd.DataFrame(list(dist_records.items()), columns=['District', 'Count']).sort_values('Count', ascending=True)
        plt.figure(figsize=(10, 6))
        plt.barh(df_dist['District'], df_dist['Count'], color='#2b5c8f')
        plt.title('Số lượng Tin đăng Thô thu thập được theo từng Quận/Huyện', fontsize=12, fontweight='bold')
        plt.xlabel('Số tin đăng')
        plt.tight_layout()
        plt.show()
else:
    print('Chưa có file metadata. Bạn có thể chạy crawler bằng lệnh: python -m src.crawler')

## 2. Kiểm toán Dữ liệu Thô (Data Completeness & Missing Rate)

In [ ]:
if INTERIM_PARQUET.exists():
    df_raw = pd.read_parquet(INTERIM_PARQUET)
elif RAW_JSONL.exists():
    records = []
    with open(RAW_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    df_raw = pd.DataFrame(records)
else:
    print('Chưa có dữ liệu thô. Vui lòng chạy crawler trước.')
    df_raw = pd.DataFrame()

if not df_raw.empty:
    print(f'Tổng số bản ghi đọc được: {len(df_raw):,}')
    audit_cols = ['price', 'size', 'deposit', 'latitude', 'longitude', 'ward_name', 'street_name', 'furnishing_rent']
    available_cols = [c for c in audit_cols if c in df_raw.columns]
    
    missing_summary = pd.DataFrame({
        'Số dòng khuyết': df_raw[available_cols].isnull().sum(),
        'Tỷ lệ khuyết (%)': (df_raw[available_cols].isnull().sum() / len(df_raw) * 100).round(2)
    }).sort_values('Tỷ lệ khuyết (%)', ascending=False)
    
    display(missing_summary)
    
    # Biểu đồ tỷ lệ khuyết
    plt.figure(figsize=(9, 4))
    sns.barplot(x=missing_summary.index, y=missing_summary['Tỷ lệ khuyết (%)'], palette='Blues_r')
    plt.title('Tỷ lệ Dữ liệu Khuyết của các Trường cốt lõi (Trực quan hóa Tiền xử lý)')
    plt.ylabel('Tỷ lệ khuyết (%)')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()